# RNN → LSTM → Attention → T5 encoder-decoder QA on Kaggle

Train three scratch encoder-decoders and fine-tune T5-small using one shared QA workflow.

Enable **Internet** and a **GPU accelerator** in Kaggle settings. Edit `REPO_URL`
below after pushing this project to GitHub. The workflow uses GPU 0, even if
Kaggle exposes two GPUs. Long implementations live in `src/qa_assignment/`.

Artifacts go to `/kaggle/working/qa_baselines_retrain/`. Each variant and seed has its
own checkpoint directory. Kaggle working files are temporary: download the
archive or save the notebook outputs, then attach those outputs to resume a
later session. Checkpoints are written every 250 optimizer steps and each epoch.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/NhanPhan090806/Mamba_Attention"  # fill in your GitHub repository URL after pushing the project
REPO_REF = "main"  # preferably the same commit SHA in BOTH notebooks
SOURCE_SUBDIR = "."  # edit if this assignment lives below the repository root
WORKSPACE = Path("/kaggle/working")
CLONE_ROOT = WORKSPACE / "qa_assignment_source"
OUTPUT_ROOT = WORKSPACE / "qa_baselines_retrain"
RESTORE_FROM = None  # e.g. Path('/kaggle/input/prior-qa-output/qa_assignment')

if not REPO_URL.strip():
    raise ValueError("Set REPO_URL to the GitHub repository you pushed this project to.")
if not WORKSPACE.is_dir():
    raise RuntimeError("This notebook is intended for Kaggle. Run local tests with pytest.")
if not (CLONE_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(CLONE_ROOT)], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_ROOT = (CLONE_ROOT / SOURCE_SUBDIR).resolve()
assert (REPO_ROOT / "pyproject.toml").is_file(), "Check SOURCE_SUBDIR."
print("Source:", REPO_ROOT)
print("Artifacts:", OUTPUT_ROOT)
subprocess.run(["git", "-C", str(CLONE_ROOT), "rev-parse", "HEAD"], check=True)

## Install the Python helpers

Run setup before importing project modules. These requirements pin the Hugging
Face interface used here and preserve Kaggle's preinstalled CUDA-enabled PyTorch.
Setup also adds the cloned `src/` directory to this running kernel's import path;
a subprocess editable install alone cannot refresh an already-started kernel.
If you previously imported Transformers in this session, restart the session
after installing and rerun from the top. A successful import is followed by an
actual forward/backward and generation probe before training.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_ROOT / "requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "-e", str(REPO_ROOT)], check=True)

# Editable-install .pth files are read at interpreter startup. This kernel
# predates the pip subprocess, so explicitly expose the cloned sources now.
import importlib

SOURCE_ROOT = (REPO_ROOT / "src").resolve()
if not (SOURCE_ROOT / "qa_assignment" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing qa_assignment sources under {SOURCE_ROOT}; check SOURCE_SUBDIR.")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()

import qa_assignment
if Path(qa_assignment.__file__).resolve().parent != SOURCE_ROOT / "qa_assignment":
    raise RuntimeError("qa_assignment was loaded from another checkout. Restart the session and rerun setup.")
print("Loaded helpers:", qa_assignment.__file__)

import torch
from qa_assignment.utils import environment_info
from qa_assignment.runtime import restore_artifacts

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator in Kaggle settings.")
DEVICE = "cuda:0"
if RESTORE_FROM is not None:
    print(restore_artifacts(RESTORE_FROM, OUTPUT_ROOT))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
environment_info(REPO_ROOT)

## Download the whole dataset and prepare identical splits

Both official SQuAD 1.1 files are downloaded in full. `MAX_TRAIN_EXAMPLES=None`
trains on all eligible training examples; set 10,000 for a shorter pilot project.
Examples are retained only when the full formatted question and context fit
the input cap. No answer-guided cropping occurs. Internal development articles
come exclusively from training, and official validation is reserved for final
reporting. Scores describe the retained subset.

Copy the exact `DataConfig` between notebooks. Compare `data_fingerprint` values
before comparing results. You can restore `data/` from previous output to reuse
tokenization, but both notebooks can independently reproduce the same split.

In [ ]:
from qa_assignment.config import DataConfig
from qa_assignment.data import prepare_data, QACollator

DATA_CONFIG = DataConfig(
    max_input_length=512,
    max_output_length=64,
    development_fraction=0.1,
    split_seed=2026,
    max_train_examples=None,
)
bundle = prepare_data(OUTPUT_ROOT / "data", DATA_CONFIG)
collator = QACollator(bundle.tokenizer, DATA_CONFIG)
print("Data fingerprint:", bundle.manifest["data_fingerprint"])
print("Prepared cache:", bundle.directory)
print("Train / development / validation:", len(bundle.train), len(bundle.development), len(bundle.validation))
bundle.manifest["statistics"]

## Four models, one dataset and metric pipeline

RNN and LSTM use PyTorch's built-in `nn.RNN` (tanh) and `nn.LSTM` for both
encoder and decoder. The encoder's final state initializes the decoder;
packed input sequences exclude right padding from that state. These two
baselines have no cross-attention. The attention model uses PyTorch SDPA
with a bidirectional self-attention encoder, causal self-attention decoder,
and decoder cross-attention. T5-small uses Hugging Face pretrained weights.

No Mamba installation or CUDA-extension compilation is needed. All four
use the same tokenizer, retained example IDs, input/output caps, article
partition, greedy generation, and metrics. Scratch models share width,
depth, optimizer settings, and sample order; parameter counts differ.
T5 has its own declared fine-tuning recipe and is labeled pretrained.
Runs execute sequentially in the order below, one model on GPU at a time.

In [ ]:
from dataclasses import replace
from qa_assignment.config import ModelConfig, TrainConfig, BenchmarkConfig, BASELINE_VARIANTS

MODEL_VARIANTS = list(BASELINE_VARIANTS)  # rnn_rnn, lstm_lstm, attn_attn, t5_small
SEEDS = [42]
PRECISION = "fp32"  # keep common; change only after every model passes its probe
SCRATCH_MODEL_CONFIG = ModelConfig(d_model=128, encoder_layers=2, decoder_layers=2,
                                   heads=4, feedforward_dim=512, dropout=0.1)
EARLY_STOPPING_PATIENCE = 0  # disabled for scratch retraining
EARLY_STOPPING_MIN_DELTA = 0.1  # absolute F1 points; scores run from 0 to 100
EARLY_STOPPING_MIN_STEPS = 0
EVAL_EVERY_STEPS = 1000  # also evaluates at epoch end; optimizer steps, not microbatches
SCRATCH_TRAIN_CONFIG = TrainConfig(epochs=5, micro_batch_size=4, accumulation_steps=4,
                                   eval_batch_size=8, learning_rate=3e-4, precision=PRECISION,
                                   warmup_fraction=0.01,
                                   save_every_steps=250, development_limit=None,
                                   eval_every_steps=EVAL_EVERY_STEPS,
                                   early_stopping_patience=EARLY_STOPPING_PATIENCE,
                                   early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
                                   early_stopping_min_steps=EARLY_STOPPING_MIN_STEPS)
T5_MODEL_CONFIG = ModelConfig(variant="t5_small", t5_name="google-t5/t5-small")
T5_TRAIN_CONFIG = replace(SCRATCH_TRAIN_CONFIG, epochs=5, learning_rate=1e-4,
                          warmup_fraction=0.05, early_stopping_patience=5,
                          early_stopping_min_steps=5000)
MODEL_CONFIGS = {variant: T5_MODEL_CONFIG if variant == "t5_small" else
                 replace(SCRATCH_MODEL_CONFIG, variant=variant) for variant in MODEL_VARIANTS}
TRAIN_CONFIGS = {variant: T5_TRAIN_CONFIG if variant == "t5_small" else
                 SCRATCH_TRAIN_CONFIG for variant in MODEL_VARIANTS}
BENCHMARK_CONFIG = BenchmarkConfig(examples=200, repeats=3,
                                   throughput_batch_size=4, fixed_output_tokens=32)
for variant in MODEL_VARIANTS:
    training = TRAIN_CONFIGS[variant]
    print(variant, "epochs:", training.epochs, "lr:", training.learning_rate,
          "effective batch:", training.micro_batch_size * training.accumulation_steps)

## Compatibility and resource pilots

Each fresh model runs forward/backward, finite-gradient, and cached/uncached
decoding checks, then a 50-step warmed training pilot. Inspect memory and
estimated epoch training time for all four before starting full runs.
Estimates exclude evaluation and checkpointing. Reduce shared microbatch
size and increase accumulation if memory requires it. These pilot models
are discarded; the main runs begin fresh or resume their own checkpoints.

In [ ]:
from qa_assignment.runtime import run_pilot
from qa_assignment.utils import write_json

PILOT_STEPS = 50
pilots = {}
for variant in MODEL_VARIANTS:
    pilots[variant] = run_pilot(bundle, MODEL_CONFIGS[variant],
                                replace(TRAIN_CONFIGS[variant], seed=SEEDS[0]),
                                collator, DEVICE, steps=PILOT_STEPS, repo_root=REPO_ROOT)
    print(variant, pilots[variant])
write_json(OUTPUT_ROOT / "results" / "baseline_pilots.json", pilots)

## Learning diagnostic before retraining

Enabled by default. Each fresh scratch model learns 16 fixed training
examples, preferring different articles. It uses FP32, zero weight decay,
disabled dropout, constant diagnostic LR 3e-3, no warmup, and at most 1,500
optimizer steps. These easier memorization settings differ from main training.
Check every 100 steps and finish once generated-answer F1 reaches 95.

Reports, predictions, gradient norms, token accuracy, immediate-EOS rate,
and plots go to `results/diagnostics/<variant>/seed_42/`. Diagnostic weights
are discarded. A failed diagnostic skips that variant's main training by
default; the next model and T5 can still run. Runtime failures still raise.
Adjust the diagnostic settings and rerun it to investigate failed models.
`REQUIRE_DIAGNOSTIC_PASS=False` explicitly permits retraining failed variants.

An optional attention pilot trains 2,000 training examples for three epochs
and evaluates 256 internal-development examples. It uses the main recipe,
writes separate artifacts, and never evaluates official validation. Inspect
its curves as exploratory evidence, not an automatic pass/fail criterion.

In [ ]:
from qa_assignment.config import DiagnosticConfig
from qa_assignment.diagnostics import (run_learning_diagnostics,
                                        diagnostic_allows_training,
                                        generalization_diagnostic)
from qa_assignment.workflow import archive_outputs

RUN_OVERFIT_DIAGNOSTIC = True
REQUIRE_DIAGNOSTIC_PASS = True
RUN_GENERALIZATION_PILOT = False  # enable to inspect the small attention pilot
DIAGNOSTIC_CONFIG = DiagnosticConfig(examples=16, steps=1500,
                                     micro_batch_size=4, eval_every_steps=100,
                                     learning_rate=3e-3, target_f1=95.0)
diagnostic_reports = {}
if RUN_OVERFIT_DIAGNOSTIC:
    try:
        diagnostic_reports = run_learning_diagnostics(
            bundle, MODEL_CONFIGS, collator, OUTPUT_ROOT, DEVICE, DIAGNOSTIC_CONFIG)
    finally:
        print("Diagnostic artifacts archived at:", archive_outputs(OUTPUT_ROOT))
if RUN_GENERALIZATION_PILOT and diagnostic_allows_training(
        "attn_attn", diagnostic_reports, REQUIRE_DIAGNOSTIC_PASS):
    pilot_report = generalization_diagnostic(
        bundle, MODEL_CONFIGS["attn_attn"], SCRATCH_TRAIN_CONFIG,
        collator, OUTPUT_ROOT, DEVICE, examples=2000,
        development_examples=256, epochs=3, repo_root=REPO_ROOT)
    print(pilot_report["state"]["history"])

## Train RNN, LSTM, attention, then fine-tune T5

Each variant and seed writes its own `checkpoints/<variant>/seed_<seed>/`
folder with resumable `last.pt`, development-selected `best.pt`, step snapshots,
configuration, manifest, and dependency versions. Matching runs resume
automatically. Final official-validation EM/F1 and identical timing workloads
use the selected checkpoint. T5 also saves the selected `hf_export/`.

The output root defaults to `/kaggle/working/qa_baselines_retrain/` so these runs have
their own artifacts. Save the notebook outputs to resume a later session.

Scratch runs use five epochs, 1% warmup, and no early stopping. T5 keeps
its five-epoch limit, 5% warmup, and F1 stopping policy (patience 5 checks,
min_delta 0.1, minimum step 5,000). Development checks occur every 1,000
optimizer steps and at epoch end. They include teacher-forced loss, token
accuracy, immediate EOS, and sample answers. F1 selects `best.pt`, with
lower development loss breaking exact F1 ties. Official validation never
selects checkpoints or stops training.

Start fresh for these updated runs. Leave `RESTORE_FROM=None` and use an
empty output root. Later resumptions of these runs require the complete
checkpoint folders, identical configuration, and the same source revision.

In [ ]:
from qa_assignment.workflow import run_experiment, archive_outputs

RUN_TRAINING = True
summaries = []
if RUN_TRAINING:
    try:
        for variant in MODEL_VARIANTS:
            if not diagnostic_allows_training(variant, diagnostic_reports, REQUIRE_DIAGNOSTIC_PASS):
                continue
            for seed in SEEDS:
                summary = run_experiment(
                    bundle, MODEL_CONFIGS[variant], replace(TRAIN_CONFIGS[variant], seed=seed),
                    OUTPUT_ROOT, device=DEVICE, resume="auto",
                    benchmark_config=BENCHMARK_CONFIG, repo_root=REPO_ROOT,
                )
                summaries.append(summary)
                print(summary)
    finally:
        print("Available checkpoints/results archived at:", archive_outputs(OUTPUT_ROOT))
else:
    print("Training disabled; set RUN_TRAINING=True after inspecting pilots.")

## Compare all four encoder-decoder models

The ordered table and combined plots include RNN, LSTM, attention, and T5.
Every run logs normalized EM/F1, training loss/time, parameter count, GPU
memory, batch-one QA latency, throughput, and forced 32-token decoding.
T5 is labeled pretrained; epochs and learning rates remain visible in the
table. This measures the declared training recipes, rather than attributing
the benefit of pretraining to architecture alone. Only the selected variants
with matching data fingerprints enter this comparison.

In [ ]:
from qa_assignment.workflow import collect_results, plot_results, archive_outputs

table = collect_results(OUTPUT_ROOT, variants=MODEL_VARIANTS)
display(table)
if not table.empty:
    figures = plot_results(table, OUTPUT_ROOT, combine_groups=True)
    import matplotlib.pyplot as plt
    plt.show()

In [ ]:
INCLUDE_DATA_IN_ARCHIVE = True  # enables reuse in the second notebook
archive_path = archive_outputs(OUTPUT_ROOT, include_data=INCLUDE_DATA_IN_ARCHIVE)
print("Download from Kaggle Output:", archive_path)
print("Checkpoint root:", OUTPUT_ROOT / "checkpoints")

## Resume in another Kaggle session

Save/download these outputs, attach the prior output as a Kaggle input, and set
`RESTORE_FROM` to the folder containing `checkpoints/`, `results/`, and optionally
`data/`. Restore before preparing new artifacts. The helper copies read-only
Kaggle inputs into writable working storage; existing folders are never overwritten.
`resume='auto'` loads each variant's `last.pt`. Keep the complete checkpoint folder,
including `best.pt`, and use identical model/data/training settings and source commit.

To start a different experiment, change `OUTPUT_ROOT`; changing settings inside an
existing run is rejected. `last.pt` resumes training; `best.pt` is selected using
internal-development F1. Load only checkpoints you created or otherwise trust.